In [0]:
from pyspark.sql.functions import *
from dotenv import load_dotevn
import os

load_dotenv()

# Azure Event Hub Configuration
event_hub_namespace = os.getenv("EVENT_HUB_HOSTNAME")
event_hub_name = os.getenv("EVENT_HUB_NAME")
event_hub_conn_str = dbutils.secrets.get(scope = "secretScope", key = "eventhub-connection")

# Storage Configuration
storage_account = os.getenv("STORAGE_ACCOUNT")

kafka_options = {
    'kafka.bootstrap.servers': f"{event_hub_namespace}:9093",
    'subscribe': event_hub_name,
    'kafka.security.protocol': 'SASL_SSL',
    'kafka.sasl.mechanism': 'PLAIN',
    'kafka.sasl.jaas.config': f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username=\"$ConnectionString\" password=\"{event_hub_conn_str}\";",
    'startingOffsets':'latest',
    'failOnDataLoss':'false'
}

# Read from eventhub
raw_df = (spark
          .readStream
          .format("kafka")
          .options(**kafka_options)
          .load()
)

# Cast data to json
json_df = raw_df.selectExpr("CAST(value AS STRING) AS raw_json")

# ADSL configuration
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(scope = "secretScope", key = "storage-connection")
)

bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/patient_flow"

# Write stream to bronze storage
(
    json_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "dbfs:/mnt/bronze/_checkpoints/partient_flow")
    .start(bronze_path)
)

In [0]:
display(spark.read.format("delta").load(bronze_path))

raw_json
"{""patient_id"": ""d0838ac2-c597-4d4b-9c64-d9f1021bc269"", ""gender"": ""Male"", ""age"": 72, ""department"": ""ICU"", ""admission_time"": ""2026-05-27T19:52:30.756078+00:00"", ""discharge_time"": ""2026-05-29T13:52:30.756078+00:00"", ""bed_id"": 166, ""hospital_id"": 6}"
"{""patient_id"": ""bb0032b3-54a7-4111-a66d-529bb1ed60a6"", ""gender"": ""Female"", ""age"": 86, ""department"": ""Surgery"", ""admission_time"": ""2026-05-28T20:52:32.008313+00:00"", ""discharge_time"": ""2026-05-29T15:52:32.008313+00:00"", ""bed_id"": 342, ""hospital_id"": 3}"
"{""patient_id"": ""a3af87b8-b403-46cb-89b3-48ab93db98b4"", ""gender"": ""Female"", ""age"": 40, ""department"": ""Emergency"", ""admission_time"": ""2026-05-28T05:52:33.009542+00:00"", ""discharge_time"": ""2026-05-28T10:52:33.009542+00:00"", ""bed_id"": 457, ""hospital_id"": 6}"
"{""patient_id"": ""4a9a4910-e1bf-453c-9076-05246688c4b2"", ""gender"": ""Male"", ""age"": 45, ""department"": ""Oncology"", ""admission_time"": ""2026-05-26T15:52:34.010394+00:00"", ""discharge_time"": ""2026-05-28T21:52:34.010394+00:00"", ""bed_id"": 400, ""hospital_id"": 2}"
"{""patient_id"": ""921cfd7f-d8d1-49d4-b015-2ea5bcab381a"", ""gender"": ""Male"", ""age"": 83, ""department"": ""Oncology"", ""admission_time"": ""2026-05-26T10:52:35.011431+00:00"", ""discharge_time"": ""2026-05-27T22:52:35.011431+00:00"", ""bed_id"": 378, ""hospital_id"": 2}"
"{""patient_id"": ""4f51e77e-830d-4b85-94d9-2ae07480999f"", ""gender"": ""Female"", ""age"": 20, ""department"": ""Emergency"", ""admission_time"": ""2026-05-26T20:52:36.012310+00:00"", ""discharge_time"": ""2026-05-27T10:52:36.012310+00:00"", ""bed_id"": 378, ""hospital_id"": 7}"
"{""patient_id"": ""672624ad-e7a0-4a75-b1f0-559bc22cac31"", ""gender"": ""Male"", ""age"": 10, ""department"": ""Maternity"", ""admission_time"": ""2026-05-27T19:52:37.013285+00:00"", ""discharge_time"": ""2026-05-30T18:52:37.013285+00:00"", ""bed_id"": 128, ""hospital_id"": 1}"
"{""patient_id"": ""a907585c-c027-4b90-8fae-4b0d75133f88"", ""gender"": ""Male"", ""age"": 53, ""department"": ""Cardiology"", ""admission_time"": ""2026-05-28T19:52:38.014644+00:00"", ""discharge_time"": ""2026-05-29T18:52:38.014644+00:00"", ""bed_id"": 372, ""hospital_id"": 4}"
"{""patient_id"": ""3feb78ed-e696-4fd0-941c-94d9034221d8"", ""gender"": ""Female"", ""age"": 43, ""department"": ""Pediatrics"", ""admission_time"": ""2026-05-28T23:52:39.015772+00:00"", ""discharge_time"": ""2026-05-30T12:52:39.015772+00:00"", ""bed_id"": 85, ""hospital_id"": 3}"
"{""patient_id"": ""3271b94e-6e6c-43b6-81f7-92d950305bcf"", ""gender"": ""Female"", ""age"": 56, ""department"": ""Maternity"", ""admission_time"": ""2026-05-30T05:52:40.016998+00:00"", ""discharge_time"": ""2026-05-29T22:52:40.016998+00:00"", ""bed_id"": 224, ""hospital_id"": 4}"
